# Revenue Forecasting — Partner Health Check Extension

Extends the existing [Partner Health Check](../README.md) trend segmentation
with a forward-looking 3-month revenue forecast, comparing two standard
methods rather than presenting a single model as ground truth.

## Why partner_type, not individual partners

Each individual partner only has ~13-18 months of history — too little to
fit a reliable trend on its own, and noisy enough that a per-partner
forecast would be more misleading than useful. Aggregating to
**partner_type** (Content Site, Influencer, Coupon Site, Affiliate,
Loyalty/Cashback) gives a more stable signal while still being actionable —
budget and retention decisions are often made at this level anyway.

## Methods compared

1. **Simple moving average** — average of the last 3 months, carried flat
   forward. Assumes no trend, just recent typical level.
2. **Linear regression (OLS)** — fits a straight trend line through the
   full history and extrapolates it forward. Assumes the trend continues.

## How accuracy was measured

Backtested both methods: held out the last 3 months of actual data,
forecast them using only the months before, then measured Mean Absolute
Error (MAE) against what actually happened. This is a standard, honest way
to compare forecast methods — it doesn't just check that a model fits
history, it checks whether it would have predicted correctly.

## Results

| Partner Type | Months of History | Moving Avg MAE | Linear Regression MAE | Better Method |
|---|---|---|---|---|
| Affiliate | 15 | R13,857 | R14,822 | Moving Average |
| Content Site | 18 | R57,610 | R93,882 | Moving Average |
| Coupon Site | 18 | R87,034 | R80,440 | Linear Regression |
| Influencer | 18 | R28,802 | R24,824 | Linear Regression |
| Loyalty/Cashback | 15 | R10,022 | R6,714 | Linear Regression |

**No single method wins across the board.** Moving average performs better
for the two largest, most volatile categories (Affiliate, Content Site) —
their month-to-month swings are large and don't follow a clean trend, so a
simple recent-average is a safer bet than extrapolating a line through the
noise. Linear regression performs better for categories with more
consistent directional movement (Coupon Site, Influencer,
Loyalty/Cashback).

## 3-month forecast (next 3 months from last data point)

| Partner Type | Monthly Trend Slope | Recommended Forecast |
|---|---|---|
| Content Site | -R3,537/month | ~R249,000/month (moving avg — see note below) |
| Coupon Site | +R3,513/month | ~R115,000–121,000/month (regression, rising) |
| Influencer | +R2,892/month | ~R127,000–132,000/month (regression, rising) |
| Affiliate | -R1,367/month | ~R25,000/month (moving avg — declining trend, but flat carry is safer per backtest) |
| Loyalty/Cashback | +R745/month | ~R23,500–25,000/month (regression, mildly rising) |

Full month-by-month figures: `forecast_by_partner_type.csv`
Backtest detail: `forecast_backtest_accuracy.csv`
Chart (Content Site, the largest category): `forecast_chart_Content Site.png`

## Honest limitations

- This is aggregate, category-level forecasting, not partner-by-partner
  prediction — appropriate for budget/retention planning at the segment
  level, not for predicting any single partner's next invoice.
- Both methods assume the recent past is a reasonable guide to the near
  future. Neither accounts for seasonality, campaign calendar effects, or
  external shocks (a partner site redesign, a competitor's promotion,
  etc.) — with only 15-18 months of data, there isn't enough history to
  reliably detect a seasonal pattern separate from noise.
- A 3-month horizon was chosen deliberately short, since both methods'
  backtested error grows with the forecast horizon — this is not a
  12-month annual forecast and shouldn't be used as one.

## Tech

Python | pandas | scikit-learn (LinearRegression) | matplotlib
